# Phase 1: Data Cleaning, EDA & Feature Engineering

**Goal**: Prepare Amazon review dataset for recommendation modeling

**Outputs**:
- cleaned_interactions.csv
- user_features.csv
- item_features.csv
- item_metadata.csv
- popularity_baseline.csv
- Train/Val/Test splits

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
RAW_DATA_PATH = '../data/raw/amazon_reviews.csv'
PROCESSED_DIR = '../processed/'

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("✓ Imports successful")
print(f"Timestamp: {datetime.now()}")

## 2. Load Raw Data

In [ ]:
try:
    df_raw = pd.read_csv(RAW_DATA_PATH, on_bad_lines='skip')
    print(f"✓ Loaded {len(df_raw)} rows")
except Exception as e:
    print(f"✗ Error loading data: {e}")
    raise

print(f"\nDataset shape: {df_raw.shape}")
print(f"\nColumn names and types:\n{df_raw.dtypes}")
print(f"\nFirst 3 rows:\n{df_raw.head(3)}")
print(f"\nMissing values:\n{df_raw.isnull().sum()}")

## 3. Standardize Column Names

In [ ]:
# Map common Amazon dataset column names
column_mapping = {
    'reviewerID': 'user_id',
    'reviewer_id': 'user_id',
    'userId': 'user_id',
    'asin': 'product_id',
    'productId': 'product_id',
    'product_id': 'product_id',
    'overall': 'rating',
    'rating': 'rating',
    'reviewText': 'review_text',
    'review_text': 'review_text',
    'text': 'review_text',
    'summary': 'review_summary',
    'title': 'review_summary',
    'brand': 'brand',
    'category': 'category',
    'price': 'price',
    'unixReviewTime': 'timestamp',
    'timestamp': 'timestamp',
    'date': 'timestamp',
    'image_url': 'image_url',
    'images': 'image_url'
}

df = df_raw.rename(columns=column_mapping).copy()

# Keep only useful columns
useful_cols = [
    'user_id', 'product_id', 'rating', 'review_text', 
    'review_summary', 'brand', 'category', 'price', 'image_url', 'timestamp'
]

available_cols = [col for col in useful_cols if col in df.columns]
df = df[available_cols].copy()

print(f"✓ Standardized to {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")

## 4. Data Cleaning

In [ ]:
print(f"Initial rows: {len(df)}")

# Remove duplicates
df = df.drop_duplicates(subset=['user_id', 'product_id', 'timestamp'], keep='last')
print(f"After removing duplicates: {len(df)}")

# Remove rows with missing critical fields
df = df.dropna(subset=['user_id', 'product_id', 'rating'])
print(f"After removing null critical fields: {len(df)}")

# Fill missing text
df['review_text'] = df['review_text'].fillna('')
df['review_summary'] = df['review_summary'].fillna('')

# Clean text
def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    return ' '.join(text.lower().split())

df['review_text'] = df['review_text'].apply(clean_text)
df['review_summary'] = df['review_summary'].apply(clean_text)

# Convert rating to int
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df[df['rating'].notna()]
df['rating'] = df['rating'].astype(int)

# Filter valid ratings
df = df[df['rating'].between(1, 5)]
print(f"After cleaning text & ratings: {len(df)}")

# Handle price
if 'price' in df.columns:
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Handle timestamp
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['timestamp'] = pd.to_datetime(df['timestamp'].apply(
        lambda x: datetime.fromtimestamp(x) if x and x > 1000000000 else None
    ), errors='coerce')
else:
    df['timestamp'] = datetime.now()

print(f"✓ Data cleaning complete. Final shape: {df.shape}")

## 5. Exploratory Data Analysis

In [ ]:
print("="*70)
print("EXPLORATORY DATA ANALYSIS")
print("="*70)

print(f"\nDataset summary:")
print(f"  Total reviews: {len(df):,}")
print(f"  Unique users: {df['user_id'].nunique():,}")
print(f"  Unique products: {df['product_id'].nunique():,}")
if df['timestamp'].notna().sum() > 0:
    print(f"  Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

print(f"\nRating distribution:")
rating_dist = df['rating'].value_counts().sort_index()
print(rating_dist)

# Calculate sparsity
sparsity_pct = (1 - (len(df) / (df['user_id'].nunique() * df['product_id'].nunique()))) * 100
print(f"\nSparsity: {sparsity_pct:.2f}%")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Rating distribution
axes[0, 0].bar(rating_dist.index, rating_dist.values, color='steelblue')
axes[0, 0].set_title('Rating Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: Reviews per user (top 20)
reviews_per_user = df.groupby('user_id').size().sort_values(ascending=False)
axes[0, 1].barh(range(min(20, len(reviews_per_user))), reviews_per_user.head(20).values, color='coral')
axes[0, 1].set_title('Top 20 Most Active Users', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Number of Reviews')
axes[0, 1].invert_yaxis()

# Plot 3: Reviews per product (top 20)
reviews_per_product = df.groupby('product_id').size().sort_values(ascending=False)
axes[1, 0].barh(range(min(20, len(reviews_per_product))), reviews_per_product.head(20).values, color='lightgreen')
axes[1, 0].set_title('Top 20 Most Reviewed Products', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Number of Reviews')
axes[1, 0].invert_yaxis()

# Plot 4: Sparsity
axes[1, 1].text(0.5, 0.5, f"Sparsity: {sparsity_pct:.2f}%\n\n" +
                f"Users: {df['user_id'].nunique():,}\n" +
                f"Products: {df['product_id'].nunique():,}\n" +
                f"Interactions: {len(df):,}",
                ha='center', va='center', fontsize=11,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(f'{PROCESSED_DIR}01_eda_overview.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: 01_eda_overview.png")

## 6. Feature Engineering - User Features

In [ ]:
# User features
user_features = df.groupby('user_id').agg({
    'rating': ['count', 'mean', 'std', 'min', 'max'],
    'timestamp': 'max'
}).reset_index()

user_features.columns = ['user_id', 'review_count', 'avg_rating', 'rating_std',
                          'min_rating', 'max_rating', 'last_review_date']

user_features['rating_std'] = user_features['rating_std'].fillna(0)
user_features['days_since_review'] = (datetime.now() - user_features['last_review_date']).dt.days

print(f"User features created: {len(user_features)} users")
print(user_features.head())

user_features.to_csv(f'{PROCESSED_DIR}user_features.csv', index=False)
print(f"✓ Saved: user_features.csv")

## 7. Feature Engineering - Item Features

In [ ]:
# Item features
item_features = df.groupby('product_id').agg({
    'rating': ['count', 'mean', 'std'],
    'brand': 'first',
    'category': 'first',
    'price': 'first',
    'image_url': 'first'
}).reset_index()

item_features.columns = ['product_id', 'review_count', 'avg_rating', 'rating_std',
                          'brand', 'category', 'price', 'image_url']

item_features['rating_std'] = item_features['rating_std'].fillna(0)
item_features['engagement_score'] = (
    item_features['review_count'] * (item_features['avg_rating'] / 5)
)

print(f"Item features created: {len(item_features)} products")
print(item_features.head())

item_features.to_csv(f'{PROCESSED_DIR}item_features.csv', index=False)
print(f"✓ Saved: item_features.csv")

## 8. Feature Engineering - Content Features

In [ ]:
# Combined text for content-based filtering
df['combined_text'] = (
    df['review_summary'].fillna('') + ' ' +
    df['review_text'].fillna('') + ' ' +
    df['brand'].fillna('') + ' ' +
    df['category'].fillna('')
).str.strip()

# Item metadata
item_metadata = df.groupby('product_id').agg({
    'combined_text': lambda x: ' '.join(x.dropna().astype(str)),
    'review_summary': lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else '',
    'brand': 'first',
    'category': 'first',
    'price': 'first',
    'image_url': 'first'
}).reset_index()

item_metadata.columns = ['product_id', 'combined_text', 'summary', 'brand', 'category', 'price', 'image_url']

print(f"Item metadata created: {len(item_metadata)} products")
print(item_metadata[['product_id', 'brand', 'category']].head())

item_metadata.to_csv(f'{PROCESSED_DIR}item_metadata.csv', index=False)
print(f"✓ Saved: item_metadata.csv")

## 9. Create Interaction Matrix

In [ ]:
# Cleaned interactions for collaborative filtering
interactions = df[['user_id', 'product_id', 'rating']].copy()

# Remove duplicates (keep highest rating)
interactions = interactions.sort_values('rating', ascending=False).drop_duplicates(
    subset=['user_id', 'product_id'], keep='first'
)

print(f"Interaction matrix created: {len(interactions)} interactions")
print(f"Rating stats:")
print(interactions['rating'].describe())

interactions.to_csv(f'{PROCESSED_DIR}cleaned_interactions.csv', index=False)
print(f"✓ Saved: cleaned_interactions.csv")

## 10. Popularity Baseline

In [ ]:
# Popularity baseline for cold-start
popularity_baseline = item_features[['product_id', 'avg_rating', 'review_count', 'engagement_score']].copy()
popularity_baseline = popularity_baseline.sort_values('engagement_score', ascending=False)

print(f"Popularity baseline (top 10):")
print(popularity_baseline.head(10))

popularity_baseline.to_csv(f'{PROCESSED_DIR}popularity_baseline.csv', index=False)
print(f"✓ Saved: popularity_baseline.csv")

## 11. Train/Val/Test Split

In [ ]:
# Temporal split: 80% train, 10% val, 10% test
interactions = interactions.sort_values('rating').reset_index(drop=True)
total_len = len(interactions)

train_idx = int(0.8 * total_len)
val_idx = int(0.9 * total_len)

train_interactions = interactions.iloc[:train_idx]
val_interactions = interactions.iloc[train_idx:val_idx]
test_interactions = interactions.iloc[val_idx:]

print(f"Train/Val/Test split:")
print(f"  Train: {len(train_interactions)} ({100*len(train_interactions)/total_len:.1f}%)")
print(f"  Validation: {len(val_interactions)} ({100*len(val_interactions)/total_len:.1f}%)")
print(f"  Test: {len(test_interactions)} ({100*len(test_interactions)/total_len:.1f}%)")

train_interactions.to_csv(f'{PROCESSED_DIR}train_interactions.csv', index=False)
val_interactions.to_csv(f'{PROCESSED_DIR}val_interactions.csv', index=False)
test_interactions.to_csv(f'{PROCESSED_DIR}test_interactions.csv', index=False)

print(f"✓ Saved: train/val/test splits")

## 12. Summary

In [ ]:
print("\n" + "="*70)
print("PHASE 1 COMPLETE: DATA PREPARATION")
print("="*70)

artifacts = {
    "cleaned_interactions.csv": f"User-item interactions ({len(interactions)} rows)",
    "train_interactions.csv": f"Training set ({len(train_interactions)} rows)",
    "val_interactions.csv": f"Validation set ({len(val_interactions)} rows)",
    "test_interactions.csv": f"Test set ({len(test_interactions)} rows)",
    "user_features.csv": f"User features ({len(user_features)} rows)",
    "item_features.csv": f"Item features ({len(item_features)} rows)",
    "item_metadata.csv": f"Item metadata ({len(item_metadata)} rows)",
    "popularity_baseline.csv": f"Popularity baseline ({len(popularity_baseline)} rows)"
}

print("\nGenerated artifacts:")
for fname, desc in artifacts.items():
    print(f"  ✓ {fname}: {desc}")

print("\nDataset characteristics:")
print(f"  Users: {df['user_id'].nunique():,}")
print(f"  Products: {df['product_id'].nunique():,}")
print(f"  Total interactions: {len(interactions):,}")
print(f"  Sparsity: {sparsity_pct:.2f}%")
print(f"  Avg rating: {interactions['rating'].mean():.2f}")
print(f"  Rating std: {interactions['rating'].std():.2f}")

print("\n✓ Ready for Notebook 2: Collaborative Filtering")